# 03 — So sánh ba thuật toán: topic
Huấn luyện trên train, chọn bằng validation, báo cáo test. Ghi vào thư mục run mới, không đụng `artifacts/serving`.

Chọn kernel `student_feedback/.venv` có IPython và các thư viện dự án, rồi Restart Kernel → Run All. Mỗi lần chạy phần xử lý tạo thư mục kết quả mới; đường dẫn được in trong output.

Notebook này train lại 3 baseline (Naive Bayes, Logistic Regression, Linear SVM), chọn bằng Macro-F1 validation và chỉ lưu pipeline thắng cuộc. Kết quả khác SVM CT4 cải tiến hiện đang dùng trên web. Không dùng điểm test để chọn lại mô hình.

**Bản sửa import:** ô code đầu tiên thực hiện cả thiết lập đường dẫn và đọc dữ liệu/huấn luyện. Nếu vẫn thấy ô riêng bắt đầu bằng `from feedback_ai...`, hãy đóng và mở lại file để nạp bản đã sửa.


In [ ]:
from pathlib import Path
from datetime import datetime
from uuid import uuid4
import sys
import importlib.util
candidates = [base/suffix for base in [Path.cwd(), *Path.cwd().parents]
              for suffix in ('', 'student_feedback', 'student-feedback-ai/student_feedback')]
ROOT = next((p for p in candidates if (p/'src/feedback_ai').is_dir() and (p/'data/raw').is_dir()), None)
if ROOT is None:
    raise FileNotFoundError('Mở notebook từ workspace hoặc thư mục student_feedback.')
SRC = str((ROOT/'src').resolve())
if SRC not in sys.path:
    sys.path.insert(0, SRC)
import feedback_ai
print('feedback_ai:', feedback_ai.__file__)
required = ('pandas', 'numpy', 'matplotlib', 'sklearn', 'joblib')
missing = [name for name in required if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError(f'Kernel {sys.executable} thiếu {missing}. Chọn Python tại {ROOT / ".venv/Scripts/python.exe"}.')
import matplotlib
matplotlib.use('Agg', force=True)
import matplotlib.pyplot as plt
try:
    from IPython.display import display, Image
except ImportError:
    def display(value):
        print(value)
    def Image(filename):
        if not Path(filename).is_file():
            raise FileNotFoundError(filename)
        return f'Ảnh đã lưu: {filename}' 
RAW = ROOT/'data/raw'
RUN = ROOT/'outputs/notebooks'/(datetime.now().strftime('%Y%m%d-%H%M%S-%f')+'-'+uuid4().hex[:8])
print('Python:', sys.executable)
print('Project:', ROOT)
print('Kết quả:', RUN)

# Khởi tạo và import dự án trong cùng ô, không phụ thuộc ô đã chạy trước.
from feedback_ai.training.workflow import train
TASK_OUTPUT = RUN / ("topic-" + uuid4().hex[:8])
print("Đang train 3 baseline; kết quả:", TASK_OUTPUT, flush=True)
result = train("topic", RAW, TASK_OUTPUT, ROOT / "data/manifest.json")
print("Mô hình được chọn:", result["selection"]["algorithm"])


In [ ]:
import pandas as pd
rows = {name: {"validation_macro_f1": result["validation"][name]["macro_f1"], "test_accuracy": result["test"][name]["accuracy"], "test_macro_f1": result["test"][name]["macro_f1"]} for name in result["test"]}
display(pd.DataFrame(rows).T.round(4))

In [ ]:
display(Image(filename=str(TASK_OUTPUT / "confusion_matrix.png")))
print("Pipeline, metadata, comparison.json và confusion matrix:", TASK_OUTPUT)
